## ANN Regression
#### Predict the salary of customers on churn data

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [2]:
## Load the data
data = pd.read_csv("../Datasets/Churn_Modelling.csv")

print(data.shape)
data.head()

(10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
## Preprocess the data
data = data.drop(columns=["RowNumber", "CustomerId", "Surname"])

data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
## Encode categorical variable
le_gender = LabelEncoder()
data['Gender'] = le_gender.fit_transform(data['Gender'])

data['Gender']

0       0
1       0
2       0
3       0
4       0
       ..
9995    1
9996    1
9997    0
9998    1
9999    0
Name: Gender, Length: 10000, dtype: int64

In [5]:
## Encode geography cols
ohe_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = ohe_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=ohe_geo.get_feature_names_out(input_features=['Geography']))
geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [6]:
## Combine ohe encoded column with the original data
data = pd.concat([data.drop(columns=['Geography']), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [7]:
## Split the data into features and target
X = data.drop(columns=["EstimatedSalary"])
y = data["EstimatedSalary"]

In [8]:
X.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,0,0.0,0.0,1.0


In [9]:
## Train test split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(8000, 12) (2000, 12) (8000,) (2000,)


In [10]:
## Scale the train and test features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]], shape=(8000, 12))

In [11]:
## Save encoder, scaler as pickle file
with open(file="../artifacts/gender_le_reg.pkl", mode='wb') as file:
    pickle.dump(obj=le_gender, file=file)

with open(file="../artifacts/geo_ohe_reg.pkl", mode='wb') as file:
    pickle.dump(obj=ohe_geo, file=file)

with open(file="../artifacts/scaler_reg.pkl", mode='wb') as file:
    pickle.dump(obj=scaler, file=file)


## ANN Implementation for Regression

In [12]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense

In [13]:
## ANN Architecuter
model = Sequential([
    Dense(units=64, activation='relu', input_dim=X_train_scaled.shape[1]),
    Dense(32, activation='relu'),
    Dense(1)    # output layer (default activation linear_activation)
])

model.summary()

e:\AI\03 - Deep Learning\Deep-Learning-Projects\ANN\Churn-Modelling\churn_venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
## Compile the model
model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])

In [15]:
## Setup for Train the model
from keras.callbacks import EarlyStopping, TensorBoard
import datetime

## Setup tensorboard
log_dir = "../regressionlog/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [16]:
## Setup early stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [17]:
## Train the model
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorboard_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 100346.8125 - mae: 100346.8125 - val_loss: 98410.5234 - val_mae: 98410.5234
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 99290.1953 - mae: 99290.1953 - val_loss: 96322.1719 - val_mae: 96322.1719
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 95729.9922 - mae: 95729.9922 - val_loss: 91169.7500 - val_mae: 91169.7500
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 88951.1016 - mae: 88951.1016 - val_loss: 82907.5547 - val_mae: 82907.5547
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 79456.3125 - mae: 79456.3125 - val_loss: 72807.8906 - val_mae: 72807.8906
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 69062.4922 - mae: 69062.4922 - val_loss: 63309.7148 - val_mae: 63309.7148
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 60174.4023 - mae: 60174.4023 - val_loss: 56261.7031 - val_mae: 56261.7031
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

### Evaluate the model

In [19]:
## Load the Tensorboard Extension
%load_ext tensorboard

## It is a Jupyter/IPython magic command that loads the TensorBoard notebook extension into the current notebook session.

In [20]:
## Now opens the TensorBoard dashboard for the log folder
%tensorboard --logdir ../regressionlog/fit

## See the output of this cell in the browser "http://localhost:6006"

## Stop the process
# !tasklist | findstr "tensorboard"
# !taskkill /F /PID <PID>

#### Stop the process (tensorboard)

In [21]:
!tasklist | findstr "tensorboard"

tensorboard.exe              28512 Console                    3      5,384 K


In [22]:
!taskkill /F /PID 28512

SUCCESS: The process with PID 28512 has been terminated.


#### Evaluate model on the test data

In [25]:
test_loss, test_mae = model.evaluate(X_test_scaled, y_test)
print(f"Test MAE : {test_mae: .4f}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 50348.1836 - mae: 50348.1836
Test MAE :  50348.1836


In [26]:
## Save the model
model.save("../artifacts/reg_model.h5")